In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd

In [0]:
df = spark.read.table("workspace.default.bronze_scada")
display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
w = Window.orderBy("date")

silver = (
    df.select(
    F.to_timestamp(F.col("Date/Time"), "dd MM yyyy HH:mm").alias("date"),
    F.col("LV ActivePower (kW)").alias("puissance"),
    F.col("Wind Speed (m/s)").alias("vent"),
    F.col("Theoretical_Power_Curve (KWh)").alias("puissance_theorique"),
    F.col("Wind Direction (°)").alias("direction")
)
    .withColumn("ecart_min", (F.col("date").cast("long") - F.lag("date", 1).over(w).cast("long")) / 60)
    .withColumn("puissance", F.greatest(F.col("puissance"), F.lit(0)))
    .withColumn("arret",((F.col("puissance") == 0) & (F.col("puissance_theorique") > 0)).cast("int"))
    .withColumn("sous_perf", ((F.col("puissance") > 0) & 
                (F.col("puissance") < 0.7 * F.col("puissance_theorique"))).cast("int"))  
    .withColumn("apres_trou", (F.col("ecart_min") > 10).cast("int"))
    .withColumn("flag", F.when(F.col("arret") == 1, "arret")
                    .when(F.col("sous_perf") == 1, "sous-perf"))
    .withColumn("mois", F.month("date"))                                       
)
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_scada")

display(silver)

In [0]:
%sql
SELECT sum(arret) AS NB_arret
FROM silver_scada

In [0]:
%sql
SELECT sum(sous_perf) AS NB_sous_perf
FROM silver_scada

In [0]:
%sql
SELECT sum(apres_trou) AS NB_de_trou
FROM silver_scada

In [0]:
%sql
SELECT date, ecart_min/60 AS Heure_manquante
FROM silver_scada
where ecart_min > 10
ORDER BY Heure_manquante DESC

In [0]:
%sql
SELECT MIN(CASE WHEN puissance_theorique > 0 THEN vent END) AS cut_in
FROM silver_scada

In [0]:
completude = (
    silver
    .groupBy(F.month("date").alias("mois"))
    .agg(
        F.count("*").alias("nb_mesures"),
        F.max(F.dayofmonth(F.last_day("date"))).alias("nb_jours")
    )
    .withColumn("nb_attendu", F.col("nb_jours") * 144)
    .withColumn("nb_manquant", F.col("nb_attendu") - F.col("nb_mesures"))
    .withColumn("completude_pct", F.round(100 * F.col("nb_mesures") / F.col("nb_attendu"), 1))
    .orderBy("mois")
)
display(completude)

In [0]:
display(silver)

Databricks visualization. Run in Databricks to view.

In [0]:
total = silver.count()

distribution = (
    silver
    .withColumn("vent_bin", F.floor(F.col("vent") * 2) / 2)
    .groupBy("vent_bin")
    .agg(F.count("*").alias("nb_mesures"))
    .withColumn("pct", F.round(100 * F.col("nb_mesures") / total, 2))
    .orderBy("vent_bin")
)

cut_in = 3.0
pdf = distribution.toPandas()

couleurs = ["lightgray" if v < cut_in else "steelblue" for v in pdf["vent_bin"]]

plt.figure(figsize=(10, 4))
plt.bar(pdf["vent_bin"], pdf["pct"], width=0.45, align="edge", color=couleurs)
plt.axvline(cut_in, color="red", linestyle="--", label=f"Cut-in ({cut_in} m/s)")
plt.xlabel("Vitesse du vent (m/s)")
plt.ylabel("% du temps")
plt.title("Distribution des vitesses de vent en 2018")
plt.legend()
plt.show()

In [0]:
par_mois = (
    silver.groupBy("mois")
    .agg(
        F.round(F.sum("puissance") / 6 / 1000, 1).alias("energie_reelle_mwh"),
        F.round(F.sum("puissance_theorique") / 6 / 1000, 1).alias("energie_theorique_mwh"),
    )
    .withColumn("ecart_mwh",
        F.round(F.col("energie_theorique_mwh") - F.col("energie_reelle_mwh"), 1))
    .withColumn("ratio_pct",
        F.round(100 * F.col("energie_reelle_mwh") / F.col("energie_theorique_mwh"), 1))
    .orderBy("mois")
)
display(par_mois)

Databricks visualization. Run in Databricks to view.

In [0]:
SEUIL = 0.7        # sous-performance = moins de 70 % de la théorique
DUREE_MIN = 6      # 6 mesures x 10 min = 1 heure minimum

w = Window.orderBy("date")
w_cumul = w.rowsBetween(Window.unboundedPreceding, 0)

df = (
    spark.read.table("silver_scada")
    .withColumn("anomalie", F.col("flag").isNotNull().cast("int"))
)

In [0]:
df = (
    df
    .withColumn("anomalie_prec", F.coalesce(F.lag("anomalie").over(w), F.lit(0)))
    .withColumn("debut_episode",
        ((F.col("anomalie") == 1) &
         ((F.col("anomalie_prec") == 0) | (F.col("apres_trou") == 1))).cast("int"))
    .withColumn("episode_id", F.sum("debut_episode").over(w_cumul))
)

df = (
    df
    .withColumn("duree_episode",
        F.count("*").over(Window.partitionBy("episode_id", "anomalie")))
    .withColumn("anomalie_confirmee",
        ((F.col("anomalie") == 1) & (F.col("duree_episode") >= DUREE_MIN)).cast("int"))
    .withColumn("perte_mwh",
        F.when(F.col("anomalie_confirmee") == 1,
               F.greatest(F.col("puissance_theorique") - F.col("puissance"), F.lit(0)) / 6 / 1000)
         .otherwise(0.0))
)

In [0]:
gold_mois = (
    df.groupBy("mois")
    .agg(
        F.round(F.sum("puissance_theorique") / 6 / 1000, 1).alias("energie_theorique_mwh"),
        F.round(F.sum("puissance") / 6 / 1000, 1).alias("energie_reelle_mwh"),
        F.round(F.sum(F.when(F.col("flag") == "arret", F.col("perte_mwh")).otherwise(0)), 1).alias("perte_arret_mwh"),
        F.round(F.sum(F.when(F.col("flag") == "sous_performance", F.col("perte_mwh")).otherwise(0)), 1).alias("perte_sous_perf_mwh"),
        F.round(F.sum("perte_mwh"), 1).alias("perte_totale_mwh"),
        F.round(F.sum("anomalie_confirmee") / 6, 1).alias("heures_anomalie"),
    )
    .withColumn("perte_pct",
        F.round(100 * F.col("perte_totale_mwh") / F.col("energie_theorique_mwh"), 1))
    .orderBy("mois")
)
gold_mois.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_pertes_mensuelles")
display(gold_mois)

In [0]:
gold_episodes = (
    df.filter(F.col("anomalie_confirmee") == 1)
    .groupBy("episode_id")
    .agg(
        F.min("date").alias("debut"),
        F.max("date").alias("fin"),
        F.round(F.count("*") / 6, 1).alias("duree_h"),
        F.round(F.avg("vent"), 1).alias("vent_moy"),
        F.round(100 * F.avg((F.col("flag") == "arret").cast("int")), 0).alias("part_arret_pct"),
        F.round(F.sum("perte_mwh"), 2).alias("perte_mwh"),
    )
    .orderBy(F.col("perte_mwh").desc())
)
gold_episodes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_episodes_anomalies")
display(gold_episodes)